# Notebook 4: Baselines and Evaluation
**DS340W Capstone — Group 60**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, torch.nn as nn
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings; warnings.filterwarnings('ignore')
base_path = '/content/drive/MyDrive/capstone_project'

test_pred_hybrid = np.load(f'{base_path}/processed_data/test_predictions.npy')
test_pred_stl = np.load(f'{base_path}/processed_data/test_predictions_stl.npy')
test_actual = np.load(f'{base_path}/processed_data/test_actual.npy')
train_df = pd.read_csv(f'{base_path}/processed_data/train_data.csv'); train_df['date'] = pd.to_datetime(train_df['date'])
test_df = pd.read_csv(f'{base_path}/processed_data/test_data.csv'); test_df['date'] = pd.to_datetime(test_df['date'])
precinct_ids = np.load(f'{base_path}/processed_data/precinct_ids.npy')
CRIME_TYPES = ['assault', 'criminal_damage', 'robbery', 'theft']
FEAT_COLS = ['weekend', 'holiday', 'temperature', 'humidity', 'wind_speed', 'precipitation']
print(f"Loaded. Test shape: {test_actual.shape}")

In [ ]:
# Baseline models
def run_baseline(cls, name, **kw):
    preds = np.zeros_like(test_actual)
    for p_idx, pct in enumerate(precinct_ids):
        tr = train_df[train_df['precinct']==pct].sort_values('date')
        te = test_df[test_df['precinct']==pct].sort_values('date')
        if len(te) == 0: continue
        feats = [f for f in FEAT_COLS if f in tr.columns and f in te.columns]
        X_tr, X_te = tr[feats].fillna(0).values, te[feats].fillna(0).values
        for c_idx, crime in enumerate(CRIME_TYPES):
            m = cls(**kw); m.fit(X_tr, tr[crime].values)
            preds[:len(te), p_idx, c_idx] = np.maximum(m.predict(X_te), 0)
    return preds

print("Running baselines...")
pred_lr = run_baseline(LinearRegression, "LR")
pred_ridge = run_baseline(Ridge, "Ridge", alpha=1.0)
pred_rf = run_baseline(RandomForestRegressor, "RF", n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
print("Done!")

In [ ]:
# LSTM Baseline
class LSTMBaseline(nn.Module):
    def __init__(self): super().__init__(); self.lstm = nn.LSTM(4, 32, 1, batch_first=True); self.fc = nn.Linear(32, 4)
    def forward(self, x): return self.fc(self.lstm(x)[0][:,-1,:])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_dates_s = sorted(train_df['date'].unique())
train_crimes = np.zeros((len(train_dates_s), len(precinct_ids), 4))
tr_idx = train_df.set_index(['date', 'precinct'])
for d, date in enumerate(train_dates_s):
    for p, pct in enumerate(precinct_ids):
        try:
            row = tr_idx.loc[(date, pct)]
            for c, crime in enumerate(CRIME_TYPES): train_crimes[d,p,c] = row[crime]
        except: pass

pred_lstm = np.zeros_like(test_actual)
print("Training LSTM...")
for p_idx in range(len(precinct_ids)):
    series = train_crimes[:, p_idx, :]
    X = torch.FloatTensor(np.array([series[i-7:i] for i in range(7, len(series))])).to(device)
    Y = torch.FloatTensor(np.array([series[i] for i in range(7, len(series))])).to(device)
    m = LSTMBaseline().to(device); opt = torch.optim.Adam(m.parameters(), lr=0.001)
    m.train()
    for _ in range(30): opt.zero_grad(); nn.MSELoss()(m(X), Y).backward(); opt.step()
    m.eval()
    with torch.no_grad():
        seq = torch.FloatTensor(train_crimes[-7:, p_idx, :]).unsqueeze(0).to(device)
        for t in range(test_actual.shape[0]):
            p = m(seq).cpu().numpy()[0]; pred_lstm[t, p_idx] = np.maximum(p, 0)
            seq = torch.cat([seq[:,1:,:], torch.FloatTensor(p).unsqueeze(0).unsqueeze(0).to(device)], dim=1)
    if (p_idx+1) % 10 == 0: print(f"  {p_idx+1}/{len(precinct_ids)}")
print("LSTM done!")

In [ ]:
# Full comparison
models = {'Linear Regression': pred_lr, 'Ridge Regression': pred_ridge, 'Random Forest': pred_rf,
          'LSTM': pred_lstm, 'Informer+ST-GCN': test_pred_hybrid, 'Informer+ST-GCN + STL (Ours)': test_pred_stl}

print("=" * 80)
print("FINAL RESULTS")
print("=" * 80)
for name, preds in models.items():
    print(f"\n{name}:")
    for c, crime in enumerate(CRIME_TYPES):
        y = test_actual[:,:,c].flatten(); yp = preds[:,:,c].flatten()
        print(f"  {crime:20s} MAE:{mean_absolute_error(y,yp):6.2f}  RMSE:{np.sqrt(mean_squared_error(y,yp)):6.2f}  R2:{r2_score(y,yp):6.2f}")

results = []
for name, preds in models.items():
    for c, crime in enumerate(CRIME_TYPES):
        y = test_actual[:,:,c].flatten(); yp = preds[:,:,c].flatten()
        results.append({'Model':name, 'Crime':crime, 'MAE':round(mean_absolute_error(y,yp),2),
                        'RMSE':round(np.sqrt(mean_squared_error(y,yp)),2), 'R2':round(r2_score(y,yp),2)})
pd.DataFrame(results).to_csv(f'{base_path}/outputs/final_comparison_with_stl.csv', index=False)
print("\nResults saved!")

In [ ]:
# MAE by precinct
colors_p = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for c, crime in enumerate(CRIME_TYPES):
    ax = axes[c]
    mae_p = np.mean(np.abs(test_actual[:,:,c] - test_pred_stl[:,:,c]), axis=0)
    ax.bar(range(len(precinct_ids)), mae_p, color=colors_p[c], alpha=0.7)
    ax.set_title(crime.replace('_',' ').title(), fontweight='bold')
    ax.set_xlabel('Precinct Index'); ax.set_ylabel('MAE')
plt.suptitle('MAE by Precinct (STL Enhanced Model)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{base_path}/outputs/mae_by_precinct.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Time series plots
sample_idx = [0, 10, 20, 30, 40, 48] if len(precinct_ids) > 48 else list(range(min(6, len(precinct_ids))))
for c, crime in enumerate(CRIME_TYPES):
    fig, axes = plt.subplots(2, 3, figsize=(18, 8)); axes = axes.flatten()
    for idx, p in enumerate(sample_idx[:6]):
        ax = axes[idx]
        ax.plot(range(1,8), test_actual[:,p,c], 'b-o', label='Actual', markersize=5)
        ax.plot(range(1,8), test_pred_stl[:,p,c], 'r--s', label='Predicted', markersize=5)
        ax.set_title(f'Precinct {precinct_ids[p]}', fontweight='bold')
        ax.set_xlabel('Day'); ax.set_ylabel('Count'); ax.legend(fontsize=8); ax.set_xticks(range(1,8))
    plt.suptitle(f'{crime.replace("_"," ").title()} Predictions vs Actual', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{base_path}/outputs/timeseries_{crime}.png', dpi=150, bbox_inches='tight')
    plt.show()
print("All plots saved!")

In [ ]:
# Bar charts
for c, crime in enumerate(CRIME_TYPES):
    at = test_actual[:,:,c].sum(axis=0); pt = test_pred_stl[:,:,c].sum(axis=0)
    si = np.argsort(at)[::-1][:20]
    fig, ax = plt.subplots(figsize=(14,6)); x = np.arange(len(si)); w = 0.35
    ax.bar(x-w/2, at[si], w, label='Actual', color='steelblue', edgecolor='black', linewidth=0.5)
    ax.bar(x+w/2, pt[si], w, label='Predicted', color='coral', edgecolor='black', linewidth=0.5)
    ax.set_title(f'{crime.replace("_"," ").title()} Actual vs Predicted (Top 20)', fontsize=14, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels([str(precinct_ids[i]) for i in si], rotation=45); ax.legend()
    plt.tight_layout()
    plt.savefig(f'{base_path}/outputs/barchart_{crime}.png', dpi=150, bbox_inches='tight')
    plt.show()
print("All bar charts saved!")